# Notebook for running pyClone on the S3 SNV calls

pyClone need as input a tsv file with the following columns
- mutation_id: A unique identifier for the mutation. This should be the same across datasets.
- ref_counts: The number of reads overlapping the locus matching the reference allele.
- var_counts: The number of reads overlapping the locus matching the variant allele.
- normal_cn: The copy number of the locus in non-malignant cells. This should generally be 2 except for sex chromosomes in males.
- minor_cn: The copy number of the minor allele in the malignant cells. This must be less than equal the value in the major_cn column.
- major_cn: The copy number of the major allele in the malignant cells. This should be greater than equal to the value in the minor_cn column and greater than 0.

(Copied from https://github.com/Roth-Lab/pyclone)

So one input file per sample

#### Make tsv input files from vcf

I dont have CNV calls yet, assume a normal CN of 2 (1/1)

In [1]:
import pandas as pd
import cyvcf2
import os

In [9]:
outdir = "/home/hain/EMBL/Saturn3/data/input_for_pyclone"
workdir = "/home/hain/EMBL/Saturn3/data/snv_calls/merged"
vcf_file_path = "/home/hain/EMBL/Saturn3/data/snv_calls/merged/DRUFU.vcf.gz"
patient_name = "DRUFU"

In [5]:
!bcftools query -f '%CHROM\t%POS\t%REF\t%ALT[\t%AD]\n' {vcf_file_path} > {os.path.join(outdir, "DRUFU.AD.tsv")}

In [10]:
samples = cyvcf2.VCF(vcf_file_path).samples

In [25]:
query_df = pd.read_csv(os.path.join(outdir, "DRUFU.AD.tsv"), sep="\t", header=None, names=["CHROM", "POS", "REF", "ALT"] + cyvcf2.VCF(vcf_file_path).samples).head().replace(to_replace={".": "0,0"})

In [26]:
### build variant name
query_df["mutation_id"] = query_df["CHROM"].astype(str) + "_" + query_df["POS"].astype(str) + "_" + query_df["REF"].astype(str) + "_" + query_df["ALT"].astype(str) # type: ignore

In [27]:
### set CN columns to 2 for all samples for now
query_df["normal_cn"] = 2
query_df["minor_cn"] = 1
query_df["major_cn"] = 1

In [28]:
### split AD into ref and var counts
for s in samples:
    query_df[f"{s}_ref_counts"] = query_df[s].str.split(",").str[0].astype(int)
    query_df[f"{s}_var_counts"] = query_df[s].str.split(",").str[1].astype(int)

In [29]:
query_df

,CHROM,POS,REF,ALT,DRUFU_metastasis01,DRUFU_metastasis02,DRUFU_metastasis03,DRUFU_metastasis04,DRUFU_metastasis05,DRUFU_metastasis06,...,DRUFU_metastasis03_ref_counts,DRUFU_metastasis03_var_counts,DRUFU_metastasis04_ref_counts,DRUFU_metastasis04_var_counts,DRUFU_metastasis05_ref_counts,DRUFU_metastasis05_var_counts,DRUFU_metastasis06_ref_counts,DRUFU_metastasis06_var_counts,DRUFU_metastasis07_ref_counts,DRUFU_metastasis07_var_counts
0,chr1,889970,C,T,"0,0","33,10","0,0","0,0","0,0","0,0",...,0,0,0,0,0,0,0,0,0,0
1,chr1,1198639,G,A,"33,13","26,11","23,8","30,18","0,0","32,6",...,23,8,30,18,0,0,32,6,27,11
2,chr1,1428492,C,T,"27,5","20,12","25,8","28,13","0,0","29,6",...,25,8,28,13,0,0,29,6,29,17
3,chr1,1867914,T,TTTA,"20,5","25,15","31,3","26,21","0,0","34,11",...,31,3,26,21,0,0,34,11,19,10
4,chr1,2202016,CAA,C,"0,0","21,8","0,0","0,0","0,0","0,0",...,0,0,0,0,0,0,0,0,0,0


In [30]:
### write into files
for s in samples:
    with open(f"{os.path.join(outdir, f'{patient_name}_{s}_pyclone_input.tsv')}", "w") as f:
        f.write("mutation_id\tref_counts\tvar_counts\tnormal_cn\tminor_cn\tmajor_cn\n")
        for _, row in query_df.iterrows():
            if row[s] == "0,0":
                continue
            f.write(f"{row['mutation_id']}\t{row[f'{s}_ref_counts']}\t{row[f'{s}_var_counts']}\t{row['normal_cn']}\t{row['minor_cn']}\t{row['major_cn']}\n")